# 02 — Visualización de attention con BertViz (opcional)

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase02/notebooks/02_attention_bertviz.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivo.** Ver con tus propios ojos los pesos de atención de un modelo BERT real, sobre la oración del slide:

> "el banco quebró porque estaba podrido"

Vas a poder navegar por capas y cabezas de atención, y ver cómo cada token se "conecta" con los demás.

**Requisitos.**
- ~680 MB de descarga (modelo BERT multilingual). En Colab no es problema.
- BertViz funciona mejor en Jupyter clásico/Colab; en VSCode puede no renderizar el HTML interactivo.


In [ ]:
%pip install --quiet transformers torch bertviz


## Cargar el modelo y tokenizar


In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from bertviz import head_view

MODEL_NAME = "bert-base-multilingual-cased"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

ORACION = "el banco quebró porque estaba podrido"

inputs = tokenizer.encode_plus(ORACION, return_tensors="pt", add_special_tokens=True)
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:", tokens)


## Forward pass + extracción de attention


In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

attention = outputs.attentions  # tupla de tensores (uno por capa)
print(f"Capas: {len(attention)}")
print(f"Cabezas por capa: {attention[0].shape[1]}")
print(f"Forma de cada tensor: {attention[0].shape}  (batch, heads, seq, seq)")


## Visualización interactiva

Si estás en Colab/Jupyter, esto abre un widget interactivo donde podés:
- Seleccionar capa y cabeza.
- Ver qué tokens "miran" a cuáles.


In [ ]:
head_view(attention, tokens)


## Visualización fallback: heatmap matplotlib

Si BertViz no renderiza (por ejemplo en VSCode), este heatmap muestra todas las cabezas de la primera capa.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

attn_layer0 = attention[0][0].numpy()  # (heads, seq, seq)
n_heads = min(6, attn_layer0.shape[0])

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.suptitle(f'Attention - capa 0, primeras {n_heads} cabezas\n"{ORACION}"', fontsize=11)

for h, ax in enumerate(axes.flat):
    if h >= n_heads:
        ax.axis("off")
        continue
    im = ax.imshow(attn_layer0[h], cmap="Blues", vmin=0, vmax=attn_layer0[h].max())
    ax.set_title(f"Cabeza {h+1}", fontsize=9)
    ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens, fontsize=7)

plt.tight_layout()
plt.show()

print("-> Cada celda (i, j) = cuánto el token i presta atención al token j.")
print("-> Distintas cabezas capturan relaciones distintas (sintácticas, semánticas, co-referencia).")


## Para experimentar

- Cambiá `ORACION` y observá cómo cambian los patrones.
- Pasá una oración con polisemia ("el banco subió la tasa") y compará con la actual.
- Mirá la última capa (`attention[-1]`) — suele tener atención más "semántica" que la primera.
